# 01. Librerie

In [54]:
%reset -f
import numpy as np
import pandas as pd

# 02. Input

In [ ]:
# path di input
input = r"Inserire path di input"

# Input MPS
df = pd.read_excel(input + r"\ESRS_NACE_RESULT_12_2025 del 23_12_25.xlsx", sheet_name="Sheet0", skiprows=1, header=1)
# Nuovi score G1.1
Score_G_Whis = pd.read_excel(input + r"\MPS KRI S&G_G1.2_Whistleblowing.xlsx", sheet_name="Calcolo nuovo Score G1.2", usecols=range(7))
# Nuovi score S
Score_S = pd.read_excel(input + r"\MPS_KRI S&G_S2.1_Condizioni di lavoro.xlsx", sheet_name="Confronto Score", usecols=range(7))
# Nuovi score G1.6
Score_G_Corr = pd.read_excel(input + r"\MPS KRI S&G_G1.6_Corruzione.xlsx", sheet_name="Calcolo nuovo score G1.6")

In [56]:
# Portafoglio gruppo
ptf_gruppo = pd.read_excel(input + r"\Ptf_Gruppo_MPS&MB.xlsx", sheet_name="Sheet1", dtype={"NACE_GMPS": str})

# 03. Aggancio nuovi score

## Aggancio score Whistleblowing e Corruzione

### Aggancio nuovi score Whistleblowing

In [57]:
df["NACE3"] = df["NACE3"].astype(str).str.strip()
Score_G_Whis["NACE3"] = Score_G_Whis["NACE3"].astype(str).str.strip()

# Creo un mapping fra NACE e Score_pesato
map_score_pesato = (
    Score_G_Whis.dropna(subset=["NACE3", "SCORE_PESATO"])
           .drop_duplicates(subset=["NACE3"])
           .set_index("NACE3")["SCORE_PESATO"])

# Per ST_ID = G1.2 uso SCORE_PESATO, altrimenti lascio df["ST"]
df["NEW_SCORE"] = np.where(
    df["ST_ID"].eq("G1.2"),
    df["NACE3"].map(map_score_pesato),
    df["ST"])

### Aggancio nuovi score Corruzione

In [58]:
df["NACE3"] = df["NACE3"].astype(str).str.strip()
Score_G_Corr["NACE3"] = Score_G_Corr["NACE3"].astype(str).str.strip()

# Mapping NACE 3-digit Score Corruzione
map_g16 = (
    Score_G_Corr.dropna(subset=["NACE3", "G1.6"])
                .drop_duplicates(subset=["NACE3"])
                .set_index("NACE3")["G1.6"])

# Aggiorno NEW_SCORE solo per le righe con ST_ID = G1.6
df["NEW_SCORE"] = np.where(
    df["ST_ID"].eq("G1.6"),          
    df["NACE3"].map(map_g16),
    df["NEW_SCORE"])

### Calcolo score finale "G"

In [59]:
g_codes = ["G1.1","G1.2","G1.3","G1.5","G1.6"]
nace3_exact = {"M72","M74","A01","A02","A03","M73","M69","N79","R93"}

cond_nace = (
    df["NACE3"].isin(nace3_exact)
    | df["NACE3"].str.startswith(("Q","I","C","G"))
    | (df["NACE3"].str.startswith("R") & (df["NACE3"] != "R93")))

# New score a 0 per i NACE3 selezionati
df.loc[cond_nace & df["ST_ID"].eq("G1.1"), "NEW_SCORE"] = 0

df_G = (df.loc[df["ST_ID"].isin(g_codes), ["NACE3","ST_ID","NEW_SCORE"]]
          .pivot_table(index="NACE3", columns="ST_ID", values="NEW_SCORE", aggfunc="first")
          .reindex(columns=g_codes)
          .apply(pd.to_numeric, errors="coerce"))


# Pesi
weights = (df.loc[df["ST_ID"].isin(g_codes), ["ST_ID","ST_WGT"]]
             .drop_duplicates("ST_ID")
             .set_index("ST_ID")["ST_WGT"]
             .pipe(pd.to_numeric, errors="coerce")
             .reindex(g_codes))

# New Score G
df_G["SCORE_G_NEW"] = df_G.mul(weights, axis=1).sum(axis=1) / weights.sum()

df_G

ST_ID,G1.1,G1.2,G1.3,G1.5,G1.6,SCORE_G_NEW
NACE3,,,,,,
A01,0.0,0.00,0.5,0.50000,0.500,0.300000
A02,0.0,0.00,0.0,0.70000,0.500,0.240000
A03,0.0,0.00,0.0,0.83333,0.500,0.266666
B05,1.0,0.75,0.0,1.00000,0.250,0.600000
B06,1.0,0.75,0.0,1.00000,0.250,0.600000
...,...,...,...,...,...,...
S95,0.5,0.75,0.0,0.00000,0.500,0.350000
S96,0.5,0.75,0.0,0.00000,0.500,0.350000
T97,0.5,0.00,0.0,0.00000,0.125,0.125000


### Aggancio score finale al df di partenza

In [60]:
# normalizzo NACE3
df["NACE3"] = df["NACE3"].astype(str).str.strip()
map_score_g_new = df_G["SCORE_G_NEW"].dropna()

# aggiorno SOLO le righe con ST_ID che inizia per "G"
mask_G = df["ST_ID"].str.startswith("G")
df.loc[mask_G, "KRI_VAL_NEW"] = df.loc[mask_G, "NACE3"].map(map_score_g_new)

## Aggancio score Social

### Aggancio nuovi score sub-sub-topic 2.1

## 

In [61]:
Score_S["NACE3"] = Score_S["NACE3"].astype(str).str.strip()

# Mappatura NACE 3 digit nuovo score per subtopic S2.1
map_s21 = (
    Score_S.dropna(subset=["NACE3", "SCORE_PESATO"])
           .drop_duplicates(subset=["NACE3"])
           .set_index("NACE3")["SCORE_PESATO"])

# Aggiorno NEW_SCORE solo per ST_ID = S2.1
mask_s21 = df["ST_ID"].astype(str).str.strip().eq("S2.1")

df.loc[mask_s21, "NEW_SCORE"] = df.loc[mask_s21, "NACE3"].map(map_s21)

In [62]:
s_cols = ["S2.1", "S2.2", "S2.3"]

# Creo tabella che per codice NACE3 mi da i valori di score per i tre sub-topic
df_S = (
    df.loc[df["ST_ID"].isin(s_cols), ["NACE3", "ST_ID", "NEW_SCORE"]]
      .assign(
          NACE3=lambda x: x["NACE3"].astype(str).str.strip(),
          ST_ID=lambda x: x["ST_ID"].astype(str).str.strip()
      )
      .pivot_table(
          index="NACE3",
          columns="ST_ID",
          values="NEW_SCORE",
          aggfunc="first"   
      ).reset_index())
df_S

ST_ID,NACE3,S2.1,S2.2,S2.3
0,A01,0.661234,0.7,0.798306
1,A02,0.587961,0.6,0.800000
2,A03,0.573179,0.6,0.800000
3,B05,0.466975,0.5,0.800000
4,B06,0.466975,0.5,0.800000
...,...,...,...,...
83,S95,0.134035,0.0,0.100000
84,S96,0.112532,0.0,0.100000
85,T97,0.093750,0.4,0.500000
86,T98,0.000000,0.0,0.000000


### Definizione score per sub-topic S2

In [63]:
df_S[s_cols] = df_S[s_cols].apply(pd.to_numeric, errors="coerce")

# pesi S2 dai ST_WGT
w_s2 = (
    df.loc[df["ST_ID"].isin(s_cols), ["ST_ID", "ST_WGT"]]
      .assign(ST_WGT=lambda x: pd.to_numeric(x["ST_WGT"], errors="coerce"))
      .drop_duplicates(subset=["ST_ID"])
      .set_index("ST_ID")["ST_WGT"]
      .reindex(s_cols))

w_s2 = w_s2 / w_s2.sum()

# score S2 pesato
df_S["SCORE_S2_NEW"] = df_S[s_cols].mul(w_s2, axis=1).sum(axis=1)

### Calcolo media pesata per sub-topic S2

In [64]:
df_S[s_cols] = df_S[s_cols].apply(pd.to_numeric, errors="coerce")

# pesi
w_s2 = (
    df.loc[df["ST_ID"].astype(str).str.startswith("S2", na=False), ["ST_ID", "ST_WGT"]]
      .assign(ST_WGT=lambda x: pd.to_numeric(x["ST_WGT"], errors="coerce"))
      .drop_duplicates(subset=["ST_ID"])         
      .set_index("ST_ID")["ST_WGT"]
      .reindex(s_cols))

# normalizzo i pesi 
w_s2 = w_s2 / w_s2.sum()

# media pesata
df_S["SCORE_S2_NEW"] = df_S[s_cols].mul(w_s2, axis=1).sum(axis=1)

In [65]:
# S4
s4 = (df[df["ST_ID"].str.startswith("S4", na=False)]
      .groupby("NACE3", as_index=False)["TOPIC"]
      .agg(S4="first")) 

# S3
s3 = (df[df["ST_ID"].str.startswith("S3", na=False)]
      .groupby("NACE3", as_index=False)["TOPIC"]
      .agg(S3="first"))  

# S2: precedentemente definito
s2 = df_S[["NACE3", "SCORE_S2_NEW"]].rename(columns={"SCORE_S2_NEW": "S2"})

# tabella finale
df_S_ovr = (
    s2.merge(s3, on="NACE3", how="outer")
      .merge(s4, on="NACE3", how="outer")
      .loc[:, ["NACE3", "S2", "S3", "S4"]]
)
df_S_ovr

,NACE3,S2,S3,S4
0,A01,0.719847,0.533333,0.441621
1,A02,0.662654,0.333333,0.444445
2,A03,0.657726,0.333333,0.444445
3,B05,0.588992,0.266667,0.444445
4,B06,0.588992,0.266667,0.444445
...,...,...,...,...
83,S95,0.078012,0.033333,0.166667
84,S96,0.070844,0.033333,0.166667
85,T97,0.331250,0.033333,0.222222
86,T98,0.000000,0.000000,0.000000


### Calcolo media pesata topic S

In [66]:
s_cols_ovr = ["S2", "S3", "S4"]

df_S_ovr[s_cols_ovr] = df_S_ovr[s_cols_ovr].apply(pd.to_numeric, errors="coerce")

# pesi
s_map = (
    df.loc[df["ST_ID"].astype(str).str.startswith("S"), ["ST_ID", "K_WGT"]]
      .assign(
          K_WGT=lambda x: pd.to_numeric(x["K_WGT"], errors="coerce"),
          S_MACRO=lambda x: x["ST_ID"].astype(str).str.split(".", n=1).str[0]
      )
      .groupby("S_MACRO")["K_WGT"].first())

weights_s = s_map.reindex(s_cols_ovr)
weights_s = weights_s / weights_s.sum()

# media pesata
df_S_ovr["KRI_VAL_NEW"] = (
    df_S_ovr[s_cols_ovr].mul(weights_s, axis=1).sum(axis=1))

### Aggancio al df iniziale con codice topic S aggiornato

In [67]:
df["NACE3"] = df["NACE3"].astype(str).str.strip()
df_S_ovr["NACE3"] = df_S_ovr["NACE3"].astype(str).str.strip()

# mappatura NACE3 con score topic S
map_s_overall = (
    df_S_ovr.dropna(subset=["NACE3", "KRI_VAL_NEW"])
            .set_index("NACE3")["KRI_VAL_NEW"])

mask_S = df["ST_ID"].astype(str).str.startswith("S")

# aggiorno df solo per ST_ID che iniziano per S
df.loc[mask_S, "KRI_VAL_NEW"] = df.loc[mask_S, "NACE3"].map(map_s_overall)

# 04. Aggancio al portafoglio

## Portafoglio gruppo MPS/Mediobanca

### Costruzione campione

In [68]:
# normalizzazione
ptf_gruppo["CD_SEGM_PORTAF_norm"] = ptf_gruppo["CD_SEGM_PORTAF"].where(
    ptf_gruppo["CD_SEGM_PORTAF"].isna(),
    ptf_gruppo["CD_SEGM_PORTAF"].astype(str).str.strip())

mb_set = {"Compass", "Facta", "MB", "MB INT", "Selma", "Mediobanca Premier"}

# Mappatura cluster
bins = [-np.inf, 0.15, 0.30, 0.55, 0.75, np.inf]
labels = ["molto basso", "basso", "medio", "alto", "molto alto"]

seg = ptf_gruppo["CD_SEGM_PORTAF"].astype("string").str.strip()

# Definisco perimetro MPS e MB
ptf_gruppo["Perimetro"] = (
    pd.Series("MPS", index=ptf_gruppo.index, dtype="string")
      .mask(seg.isna(), "MPS")            
      .mask(seg.isin(mb_set), "MB"))

ptf_gruppo.drop(columns="CD_SEGM_PORTAF_norm")

,CD_NGR,TX_DENOM,CD_SEGM_PORTAF,NACE_GMPS,PRO_COM_GMPS,TX_SIGLA_PRV_GMPS,GCA,Perimetro
0,100695105,IMMOBILIARE CIRIGLIONE DI LUCIANI ANNA MARIA B...,CORPORATE CENTER - ALTRO,6820,11020.0,SP,5284.23,MPS
1,92349949,EURO COSTRUZIONI SRL,CORPORATE CENTER - ALTRO,4100,61022.0,CE,120.29,MPS
2,50005074836,DIFE SRL,Compass,3821,47020.0,PT,192.00,MB
3,50013320619,IDROMACCHINE SPA,Compass,2813,27042.0,VE,529.00,MB
4,50007244054,PULIMODENA SERVIZI SRL,Compass,8100,36023.0,MO,175.00,MB
...,...,...,...,...,...,...,...,...
221878,50023512238,LA LUNA COOPERATIVA SOCIALE ET,Compass,5200,54018.0,PG,6364.43,MB
221879,50023613260,ANIMISTERIA VICENTINA SRL,Compass,2400,24013.0,VI,324.00,MB
221880,312305281,BAGNI LE PALME SRL SEMPLIFICATA,SMALL BUSINESS,9329,44066.0,AP,1013.11,MPS
221881,287102304,CARONE COSTRUZIONI SRL,SMALL BUSINESS,4120,75022.0,LE,114537.04,MPS


In [69]:
# Creo IN df una nuova colonna con NACE 2 digit ad NACE3
df["NACE3"] = df["NACE3"].astype(str).str.strip()
df["NACE2"] = df["NACE3"].str[-2:]
df["NACE2"] = df["NACE2"].astype(str).str.strip()

# definizione NACE2 da NACE_GMPS
ptf_gruppo["NACE_GMPS"] = ptf_gruppo["NACE_GMPS"].astype(str).str.strip()
ptf_gruppo["NACE2"] = ptf_gruppo["NACE_GMPS"].str[:2]

## Statistiche score Social

In [70]:
# prendo solo righe S dal df
df_S_only = df[df["ST_ID"].astype(str).str.startswith("S", na=False)].copy()

# mappatura NACE 2 digits con nuovi score Social
map_nace2_kri = (
    df_S_only.dropna(subset=["NACE2", "KRI_VAL_NEW"])
             .drop_duplicates(subset=["NACE2"])
             .set_index("NACE2")["KRI_VAL_NEW"])

# Ptf_gruppo con colonne di interesse, aggancio score social da mappatura precedente
df_ptf_kri_overall = (
    ptf_gruppo.assign(
        ESPOSIZIONE=lambda x: pd.to_numeric(x["GCA"], errors="coerce"),
        KRI_VAL_NEW=lambda x: x["NACE2"].map(map_nace2_kri),
    )[["Perimetro", "NACE2", "ESPOSIZIONE", "KRI_VAL_NEW"]])

df_ptf_kri_overall["KRI_VAL_NEW"] = pd.to_numeric(df_ptf_kri_overall["KRI_VAL_NEW"], errors="coerce")

# Definizione cluster
df_ptf_kri_overall["Distribuzione_new"] = pd.cut(
    df_ptf_kri_overall["KRI_VAL_NEW"],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True)

#  Funzione di clusterizzazione ptf
def make_cluster(df_in: pd.DataFrame, perimetro_label: str) -> pd.DataFrame:

    tot_exp = df_in["ESPOSIZIONE"].sum()
    # Totale espo no missing
    tot_exp_no_missing = df_in.loc[df_in["Distribuzione_new"].notna(), "ESPOSIZIONE"].sum()

    out = (
        df_in.groupby("Distribuzione_new", dropna=False)
             .agg(
                 N_RIGHE=("NACE2", "count"),
                 ESPOSIZIONE_CLUSTER=("ESPOSIZIONE", "sum")
             )
             .reset_index()    )

    # Percentuale standard 
    out["PERC_ESPOSIZIONE"] = out["ESPOSIZIONE_CLUSTER"] / tot_exp * 100

    # Percentuale senza missing
    out["PERC_ESPOSIZIONE_NO_MISSING"] = np.where(
        out["Distribuzione_new"].notna() & (tot_exp_no_missing != 0),
        out["ESPOSIZIONE_CLUSTER"] / tot_exp_no_missing * 100,
        np.nan
    )

    out.insert(0, "Perimetro", perimetro_label)
    return out


# Output per perimetro
df_cluster_MPS = make_cluster(df_ptf_kri_overall[df_ptf_kri_overall["Perimetro"] == "MPS"], "MPS")
df_cluster_MB  = make_cluster(df_ptf_kri_overall[df_ptf_kri_overall["Perimetro"] == "MB"],  "MB")
df_cluster_OVR = make_cluster(df_ptf_kri_overall, "OVERALL")

# Output finale
df_cluster_final = pd.concat([df_cluster_MPS, df_cluster_MB, df_cluster_OVR], ignore_index=True)
df_cluster_final


,Perimetro,Distribuzione_new,N_RIGHE,ESPOSIZIONE_CLUSTER,PERC_ESPOSIZIONE,PERC_ESPOSIZIONE_NO_MISSING
0,MPS,molto basso,60154,2.223609e+10,56.310806,56.310806
1,MPS,basso,30213,1.088164e+10,27.556727,27.556727
2,MPS,medio,18122,5.170105e+09,13.092805,13.092805
3,MPS,alto,2874,1.200306e+09,3.039662,3.039662
4,MB,molto basso,62381,7.797967e+09,48.055283,48.213305
5,MB,basso,30310,7.715899e+09,47.549536,47.705895
6,MB,medio,17473,6.569925e+08,4.048743,4.062056
7,MB,alto,195,3.031614e+06,0.018682,0.018744
8,MB,NaN,7,5.318524e+07,0.327756,NaN
9,OVERALL,molto basso,122535,3.003406e+10,53.906382,53.957890


## Statistiche score Governance

In [71]:
# Prendo solo score G
df_G_only = df[df["ST_ID"].astype(str).str.startswith("G", na=False)].copy()

# Mappatura score Governance con NACE2
map_nace2_kri_G = (
    df_G_only.dropna(subset=["NACE2", "KRI_VAL_NEW"])
             .assign(KRI_VAL_NEW=lambda x: pd.to_numeric(x["KRI_VAL_NEW"], errors="coerce"))
             .drop_duplicates(subset=["NACE2"])
             .set_index("NACE2")["KRI_VAL_NEW"])

# Ptf di interesse
df_ptf_kri_G = (
    ptf_gruppo.assign(
        ESPOSIZIONE=lambda x: pd.to_numeric(x["GCA"], errors="coerce"),
        KRI_VAL_NEW=lambda x: x["NACE2"].map(map_nace2_kri_G),
    )[["Perimetro", "NACE2", "ESPOSIZIONE", "KRI_VAL_NEW"]])

df_ptf_kri_G["KRI_VAL_NEW"] = pd.to_numeric(df_ptf_kri_G["KRI_VAL_NEW"], errors="coerce")

# Clusters
df_ptf_kri_G["Distribuzione_new"] = pd.cut(
    df_ptf_kri_G["KRI_VAL_NEW"],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True)

# Output per perimetro 
df_cluster_MPS_G = make_cluster(df_ptf_kri_G[df_ptf_kri_G["Perimetro"] == "MPS"], "MPS")
df_cluster_MB_G  = make_cluster(df_ptf_kri_G[df_ptf_kri_G["Perimetro"] == "MB"],  "MB")
df_cluster_OVR_G = make_cluster(df_ptf_kri_G, "OVERALL")

# Output finale
df_cluster_final_G = pd.concat([df_cluster_MPS_G, df_cluster_MB_G, df_cluster_OVR_G], ignore_index=True)
df_cluster_final_G


,Perimetro,Distribuzione_new,N_RIGHE,ESPOSIZIONE_CLUSTER,PERC_ESPOSIZIONE,PERC_ESPOSIZIONE_NO_MISSING
0,MPS,molto basso,16901,2.249278e+09,5.696085,5.696085
1,MPS,basso,40396,1.175135e+10,29.759201,29.759201
2,MPS,medio,41258,2.177864e+10,55.152355,55.152355
3,MPS,alto,12808,3.708868e+09,9.392359,9.392359
4,MB,molto basso,16252,1.103568e+09,6.800783,6.823146
5,MB,basso,37326,5.531402e+09,34.087486,34.199577
6,MB,medio,42900,9.148248e+09,56.376442,56.561827
7,MB,alto,13881,3.906722e+08,2.407533,2.415450
8,MB,NaN,7,5.318524e+07,0.327756,NaN
9,OVERALL,molto basso,33153,3.352846e+09,6.017829,6.023579


## Export

In [ ]:
# ---------- EXPORT ----------
out = r"Inserire path di output + File di output"
with pd.ExcelWriter(out, engine="openpyxl") as writer:

    # Dataframe Gruppo
    df_cluster_final.to_excel(writer, sheet_name="Social_Gruppo", index=False)
    df_cluster_final_G.to_excel(writer, sheet_name="Governance_Gruppo", index=False)